In [ ]:

import enum


class DemandType(enum.Enum):
    RESIDENTIAL = 0
    COMMERCIAL = 1
    INDUSTRIAL = 2


class Demand:
    type: DemandType
    
    value_current: float
    value_history: list[float] = []

    value_target: float
    population_current: int

    def __init__(self, demand_type=DemandType.RESIDENTIAL):
        self.type = demand_type
        self.value_current = 0
        self.value_target = 0
        self.value_history = []

        self.population_current = 0
        self.population_history = []
        


class DemandCollection:
    """A collection of Demand objects, accessible by index or DemandType."""

    def __init__(self, demands: list[Demand]):
        self._demands = demands

    def __getitem__(self, key):
        if isinstance(key, DemandType):
            for d in self._demands:
                if d.type == key:
                    return d
            raise KeyError(f"No demand found for type: {key}")
        return self._demands[key]

    def __iter__(self):
        return iter(self._demands)

    def __len__(self):
        return len(self._demands)

    def where(self, demand_type: DemandType) -> Demand:
        """Return the Demand matching the given DemandType."""
        return self[demand_type]


def update_demand(dmnd: Demand):
    # update history to include current demand before changing it
    dmnd.value_history.append(dmnd.value_current)


def take_census():
    print('--- Taking Census ---')

def project_population(current_population_absolute: int,
                       birth_rate: float,
                       death_rate: float,
                       migration_absolute: int) -> int:
    """
    Project the population in the iteration based on the current population, birth
    rate, death rate, and migration rate. 

    Args:
        - current_population_absolute (int): The current (residential) population.
        - birth_rate (float): The birth rate (e.g., 0.01 for 1%).
        - death_rate (float): The death rate (e.g., 0.005 for 0.5%).
        - migration_absolute (int): The absolute number of migrants.

    Returns:
        - int: The projected population for the next iteration.
    """
    projected_population = current_population_absolute
    projected_population += int(current_population_absolute * birth_rate)  # Add births
    projected_population -= int(current_population_absolute * death_rate)  # Subtract deaths
    projected_population += migration_absolute  # Add net migration
    return projected_population


def calculate_jobs_per_resident(dmnd_arr: DemandCollection) -> float:
    """
    Calculate jobs per resident based on the current demand for commercial and industrial,
    and the residential population.

    Args:
        - dmnd_arr (DemandCollection): A collection of Demand objects. Must include at least
          one residential, one commercial, and one industrial demand.
    
    Returns:
        - float: The calculated jobs per resident.
    """
    previous_commercial_pop = dmnd_arr[DemandType.COMMERCIAL].population_history[-1] if \
        len(dmnd_arr[DemandType.COMMERCIAL].population_history) > 0 else 1
    previous_industrial_pop = dmnd_arr[DemandType.INDUSTRIAL].population_history[-1] if \
        len(dmnd_arr[DemandType.INDUSTRIAL].population_history) > 0 else 1
    current_residential_pop = dmnd_arr[DemandType.RESIDENTIAL].population_current

    try:
        jobs_per_resident = (previous_commercial_pop + previous_industrial_pop) / current_residential_pop
    except ZeroDivisionError:
        print("Warning: one factor of population was zero. Caught ZeroDivisionError, returning 0 for jobs per resident (this is expected in the first iteration).")
        jobs_per_resident = 0
    return jobs_per_resident


def calculate_migration(dmnd_arr: DemandCollection, jobs_per_resident: float) -> int:
    """
    Calculate the absolute migration based on the jobs per resident and the current residential population.

    Args:
        - dmnd_arr (DemandCollection): A collection of Demand objects. Must include at least one residential demand.
        - jobs_per_resident (float): The current jobs per resident.
    Returns:
        - int: The calculated absolute migration for the next iteration.
    """
    current_residential_pop = dmnd_arr[DemandType.RESIDENTIAL].population_current
    migration_absolute = current_residential_pop * (jobs_per_resident - 1)
    return int(migration_absolute)


def main_loop(iterations=5):
    birth_rate = 0.01
    death_rate = 0.005
    migration_abs = 0
    jobs_per_resident_abs = 1

    dmnd_arr = DemandCollection([
        Demand(DemandType.RESIDENTIAL),
        Demand(DemandType.COMMERCIAL),
        Demand(DemandType.INDUSTRIAL),
    ])

    for i in range(iterations):
        print(f'--- Iteration {i} ---')
        jobs_per_resident_abs = calculate_jobs_per_resident(dmnd_arr)
        migration_abs = calculate_migration(dmnd_arr, jobs_per_resident_abs)
        proj_pop = project_population(dmnd_arr[DemandType.RESIDENTIAL].population_current,
                                      birth_rate,
                                      death_rate,
                                      migration_abs)
        for dmnd in dmnd_arr:
            print(f'Current {dmnd.type.name} Demand:', dmnd.value_current)
            update_demand(dmnd)

        take_census()


main_loop()


In [ ]:

import math
import random
from IPython.display import display
from utils.plotly_graphs.demand_graphs import plot_demand_bar, plot_demand_history


def main_loop(iterations):
    # demand_collection = DemandCollection([
    #     Demand(DemandType.RESIDENTIAL),
    #     Demand(DemandType.COMMERCIAL),
    #     Demand(DemandType.INDUSTRIAL),
    # ])

    residential_population = 100
    residential_population_history = []
    residential_demand = 40
    residential_demand_history = []
    residential_demand_target = 10
    residential_demand_target_history = []

    commercial_population = 20
    commercial_population_history = []
    commercial_demand = 0
    commercial_demand_history = []
    commercial_demand_target = 0
    commercial_demand_target_history = []

    industrial_population = 0
    industrial_population_history = []
    industrial_demand = 0
    industrial_demand_history = []
    industrial_demand_target = 0
    industrial_demand_target_history = []

    birth_rate = 0.01
    death_rate = 0.0075
    migration_rate = 0.02



    for i in range(iterations):
        # compute new targets on a 4 week cycle
        if i % 4 == 0:
            previous_residential_demand = residential_demand
            previous_commercial_demand = commercial_demand
            previous_industrial_demand = industrial_demand

            previous_residential_population = residential_population
            previous_commercial_population = commercial_population
            previous_industrial_population = industrial_population

            previous_residential_demand_target = residential_demand_target
            previous_commercial_demand_target = commercial_demand_target
            previous_industrial_demand_target = industrial_demand_target


            # calculate residential demand targets
            jobs = previous_commercial_population + previous_industrial_population
            residential_demand_target = jobs + (birth_rate - death_rate) * previous_residential_demand
            residential_demand_target = jobs + previous_residential_demand

            # calculate commercial demand targets
            commercial_demand_target = random.uniform(0, 100)

            # calculate industrial demand targets
            industrial_demand_target = random.uniform(0, 100)

        # handle population changes in every cycle
        # if there are available jobs in the city (industry + commercial population), the migration rate will increase
        # until it reaches a point where the population is sufficient to fill all available jobs, at which point migration
        # will stabilize near zero
        if industrial_population + commercial_population > (residential_population):
            migration_abs = math.ceil(residential_population * migration_rate) 
        elif industrial_population + commercial_population < (residential_population):
            migration_abs = math.ceil(residential_population * migration_rate) * -1 
        else:
            migration_abs = 0

        print(f'Jobs per resident: {(industrial_population + commercial_population) / residential_population:.2f}, Migration: {migration_abs}')

        residential_population += residential_population * birth_rate - residential_population * death_rate + migration_abs

        # compare to current, and update demand values
        def apply_demand_change(demand_value, demand_target, delta=0.1, cap=100):
            if demand_value < demand_target:
                demand_value += delta * (demand_target - demand_value)
            elif demand_value > demand_target:
                demand_value -= delta * (demand_value - demand_target)
            return min(demand_value, cap)
    
        
        residential_demand = apply_demand_change(residential_demand, residential_demand_target)
        commercial_demand = apply_demand_change(commercial_demand, commercial_demand_target)
        industrial_demand = apply_demand_change(industrial_demand, industrial_demand_target)

        def record_history(history_list, value):
            history_list.append(value)

        record_history(residential_population_history, residential_population)
        record_history(commercial_population_history, commercial_population)
        record_history(industrial_population_history, industrial_population)

        record_history(residential_demand_history, residential_demand)
        record_history(commercial_demand_history, commercial_demand)
        record_history(industrial_demand_history, industrial_demand)
        
        record_history(residential_demand_target_history, residential_demand_target)
        record_history(commercial_demand_target_history, commercial_demand_target)
        record_history(industrial_demand_target_history, industrial_demand_target)


        # display graphs for this iteration
        print(f'--- Iteration {i} ---')
        print(f'Residential Demand: {residential_demand:.2f} (Target: {residential_demand_target:.2f}). Population: {residential_population}')
        print(f'Commercial Demand: {commercial_demand:.2f} (Target: {commercial_demand_target:.2f}). Population: {commercial_population}')
        print(f'Industrial Demand: {industrial_demand:.2f} (Target: {industrial_demand_target:.2f}). Population: {industrial_population}')

    # Combined RCI over time (solid lines only)
    fig_combined = plot_demand_history(
        residential_demand_history=residential_demand_history,
        commercial_demand_history=commercial_demand_history,
        industrial_demand_history=industrial_demand_history,
        title="RCI Demand Over Time",
    )
    display(fig_combined)

    # Individual component graphs with demand + target + population
    fig_res = plot_demand_history(
        residential_demand_history=residential_demand_history,
        residential_target_history=residential_demand_target_history,
        population_history=residential_population_history,
        population_label="Residential Population",
        title="Residential Demand Over Time",
    )
    display(fig_res)

    fig_com = plot_demand_history(
        commercial_demand_history=commercial_demand_history,
        commercial_target_history=commercial_demand_target_history,
        population_history=commercial_population_history,
        population_label="Commercial Population",
        title="Commercial Demand Over Time",
    )
    display(fig_com)

    fig_ind = plot_demand_history(
        industrial_demand_history=industrial_demand_history,
        industrial_target_history=industrial_demand_target_history,
        population_history=industrial_population_history,
        population_label="Industrial Population",
        title="Industrial Demand Over Time",
    )
    display(fig_ind)

    fig_bar = plot_demand_bar(
        residential_demand, residential_demand_target,
        commercial_demand, commercial_demand_target,
        industrial_demand, industrial_demand_target,
        iteration=i,
    )
    display(fig_bar)



main_loop(100)